# Genetic Programming Notebook.
In this notebook, I follow along Burakk Anber's tutorial "[Machine Learning: Introduction to Genetic Algorithms]((https://burakkanber.com/blog/machine-learning-genetic-algorithms-part-1-javascript/)". In it, Burakk Anber develops a genetic algorithm for steering random data toward a predefined string. This is useful for cool effects, and serves as an introduction to machine learning. 

All I do is update the code to contemporary javascript/typescript, as Anber's blog post featuring the code was written 14 years ago.

## The Gene

The Gene class holds a string as code, can calculate its cost (a difference from a target string), can mutate, and mate with another string to exchange halves of code. 

In [3]:
Gene = class Gene {
  code: String = '';
  cost: Number = 9999;
  
  constructor(code: String){
    if(code) this.code = code;
  }

  random(length: Number): void {
    while(length--){
      this.code += String.fromCharCode(Math.floor(Math.random()*255));
    }
  }

  mutate(chance): void {
    if(Math.random() > chance) return;

    const index = Math.floor(Math.random() * this.code.length);
    const upOrDown = Math.random() <= 0.5 ? -1 : 1;
    const newChar = String.fromCharCode(this.code.charCodeAt(index) + upOrDown);
    let newString = '';
    for(let i=0; i < this.code.length; i++){
      if(i === index) newString += newChar;
      else newString += this.code[i];
    }

    this.code = newString;
  }

  mate(gene: Gene): Array<Gene> {
    const pivot = Math.round(this.code.length / 2) - 1;

    const child1 = this.code.substr(0, pivot) + gene.code.substr(pivot);
    const child2 = gene.code.substr(0, pivot) + this.code.substr(pivot);

    return [new Gene(child1), new Gene(child2)];
  }

  calcCost(compareTo: String): Number {
    let total = 0;
    for(let i=0; i < this.code.length; i++){
      const diff = this.code.charCodeAt(i) - compareTo.charCodeAt(i);
      total += diff * diff;
    }

    this.cost = total;
  }
}

## The Population

The Population class acts as a container for the Genes. It keeps track of the goal string that we're going to breed the Genes towards, how many generations in we are, and keeps tracks of all members (which have their code and cost).

Each generation, the list of genes is sorted (by similarity to target string), the top two Genes (the ones closest to the target string) exchange code halves, then the cells mutate, get sorted again, and the same happens for the next generation.

In [3]:
Population = class Population {
  members: Array<Gene> = [];
  goal: String;
  generationNumber = 0;
  
  constructor(goal: String, size: Number){
    this.goal = goal;
    while(size--){
      let gene = new Gene();
      gene.random(this.goal.length);
      this.members.push(gene);
    }
  }

  sort(){
    this.members.sort((a, b) => a.cost - b.cost);
  }

  generation(): void {
    this.members.forEach(member => member.calcCost(this.goal));

    this.sort();
    this.display();

    const children = this.members[0].mate(this.members[1]);
    this.members.splice(this.members.length - 2, 2, ...children);

    for(let i=0; i < this.members.length; i++){
      const member = this.members[i];
      
      member.mutate(0.5);
      member.calcCost(this.goal);
      if(member.code === this.goal){
        this.sort();
        this.display();
        return true;
      }
    }

    this.generationNumber++;
    setTimeout(() => this.generation(), 20);
  }

  display(){
    const h2 = document.createElement('h2');
    h2.innerText = `Generation ${this.generationNumber}`;
    
    const table = document.createElement('table');
    const header = document.createElement('tr');
    const codeHeader = document.createElement('td');
    const costHeader = document.createElement('td');
    codeHeader.innerText = "Code";
    codeHeader.style.width = "250px";
    costHeader.innerText = "Cost";
    header.appendChild(codeHeader);
    header.appendChild(costHeader);
    table.appendChild(header);

    this.members.forEach(member => {
      const row = document.createElement('tr');
      const codeCell = document.createElement('td');
      const costCell = document.createElement('td');
      codeCell.innerText = member.code;
      costCell.innerText = member.cost;
      row.appendChild(codeCell);
      row.appendChild(costCell);
      table.appendChild(row);
    })

    output.replaceChildren(h2, table);
  }
}

<h2>Generation 1187</h2><table><tr><td style="width: 250px;">Code</td><td>Cost</td></tr><tr><td>Hello, World!</td><td>0</td></tr><tr><td>Hello- Wnrld!</td><td>2</td></tr><tr><td>Helmo, Wormd!</td><td>2</td></tr><tr><td>Hello, Woqle!</td><td>2</td></tr><tr><td>Hello, Wormc!</td><td>2</td></tr><tr><td>Hello,!Wprlc!</td><td>3</td></tr><tr><td>Helmp, Worlc!</td><td>3</td></tr><tr><td>Hello, Woskc!</td><td>3</td></tr><tr><td>Hello, Vorkc!</td><td>3</td></tr><tr><td>Heklo- Worlc!</td><td>3</td></tr><tr><td>Helln, Wprmd!</td><td>3</td></tr><tr><td>Hdllp,!World!</td><td>3</td></tr><tr><td>Helmo, Vorlc!</td><td>3</td></tr><tr><td>Helkp, Worlc!</td><td>3</td></tr><tr><td>Hello+!Wnqld!</td><td>4</td></tr><tr><td>Helmp+ Wosld!</td><td>4</td></tr><tr><td>Helln,Wnqlc!</td><td>5</td></tr><tr><td>Helmp+ Wprlc!</td><td>5</td></tr><tr><td>Hemlp, Wopld!</td><td>6</td></tr><tr><td>Hello-!Worlb!</td><td>6</td></tr></table>

## Starting the program
Change the string "Hello, World!" in the cell below to something (not too long), and click "Run All" at the bottom of the notebook.

In [4]:
const population = new Population("Hello, World!", 20);
population.generation();